# Steps 7–8: Shape & Flux Measurement

This is the heart of the project — where galaxy shapes and fluxes are
extracted from the deblended coadd images. This step produces the
columns we ultimately need for weak lensing.

**LSST tasks:**
- `lsst.meas.base.SingleFrameMeasurementTask` (runs measurement plugins)
- `lsst.meas.base.ForcedMeasurementTask` (measure at fixed positions)

**Key measurement plugins:**
- `lsst.meas.extensions.shapeHSM` — weak lensing shapes
- `lsst.meas.modelfit` — CModel galaxy photometry
- `lsst.meas.base` — centroids, PSF fluxes, aperture fluxes

**Input:** `deepCoadd` + deblended footprints  
**Output:** `deepCoadd_meas` (source catalog with all measurements)

**Reference:** Bosch et al. (2018) §4.6–4.8

## 7.1 The Plugin Architecture

Measurement in the LSST pipeline uses a **plugin system**. Each algorithm
(centroid, flux, shape) is a plugin that registers itself and runs on
every detected source.

```python
from lsst.meas.base import SingleFrameMeasurementTask

config = SingleFrameMeasurementTask.ConfigClass()

# Enable specific plugins:
config.plugins.names = [
    'base_SdssCentroid',           # centroid
    'base_PsfFlux',                # PSF photometry
    'base_GaussianFlux',           # Gaussian-weighted flux
    'base_SdssShape',              # adaptive moments
    'ext_shapeHSM_HsmShapeRegauss',  # REGAUSS shear estimator
    'ext_shapeHSM_HsmShapeBj',       # BJ shear estimator
    'ext_shapeHSM_HsmSourceMoments', # source moments
    'ext_shapeHSM_HsmPsfMoments',    # PSF moments
    'modelfit_CModel',               # CModel galaxy photometry
]
```

### Execution order
Plugins declare dependencies. The typical order is:
1. **Centroid** (SdssCentroid) — find the center
2. **Shape** (SdssShape) — adaptive second moments (used by other plugins)
3. **PSF flux** — flux assuming point source
4. **HSM shapes** — PSF-corrected ellipticities for weak lensing
5. **CModel** — fit galaxy models for total flux

### The NoiseReplacer
Before measuring each deblended child, the `NoiseReplacer`:
1. Saves the original pixel values in the footprint region
2. Replaces all **other** children's pixels with correlated noise
3. Inserts **this** child's deblended pixels
4. Runs all measurement plugins
5. Restores the original pixels

This ensures each source is measured as if it were isolated.

## 7.2 Shape Measurement: HSM

The HSM plugins (`lsst.meas.extensions.shapeHSM`) wrap GalSim's implementation
of the Hirata-Seljak-Mandelbaum shape measurement algorithms.

### REGAUSS (Re-Gaussianization) — the default

The idea: if the PSF were perfectly Gaussian, you could deconvolve it
analytically from the adaptive moments. Real PSFs aren't Gaussian, so
REGAUSS applies a **correction for the non-Gaussian PSF**.

**Algorithm:**
1. Measure adaptive (Gaussian-weighted) second moments of the galaxy:
   $$Q_{ij} = \frac{\int I(\vec{x})\, W(\vec{x})\, x_i x_j\, d^2x}{\int I(\vec{x})\, W(\vec{x})\, d^2x}$$
   where $W$ is an elliptical Gaussian weight matched to the source shape

2. Measure the same moments for the PSF model at that position

3. Correct the galaxy moments for PSF smearing:
   $$e^{\text{corrected}} = e^{\text{galaxy}} - \frac{T_{\text{PSF}}}{T_{\text{galaxy}}} \cdot e^{\text{PSF}} + \delta e_{\text{non-Gaussian}}$$
   where $T = Q_{xx} + Q_{yy}$ is the trace (size²)

4. The non-Gaussian correction $\delta e$ is computed by comparing the
   actual PSF to a Gaussian with the same moments

**Output columns:**
```
ext_shapeHSM_HsmShapeRegauss_e1     # PSF-corrected ellipticity component 1
ext_shapeHSM_HsmShapeRegauss_e2     # PSF-corrected ellipticity component 2
ext_shapeHSM_HsmShapeRegauss_sigma  # PSF-corrected size (σ in pixels)
ext_shapeHSM_HsmShapeRegauss_flag   # True if measurement failed
```

### Resolution factor
The **resolution** $R$ quantifies how well-resolved a galaxy is:
$$R = 1 - \frac{T_{\text{PSF}}}{T_{\text{galaxy}}}$$

- $R \to 1$: galaxy much larger than PSF (well-resolved)
- $R \to 0$: galaxy same size as PSF (unresolved = star)
- For weak lensing, typically require $R > 0.3$

## 7.3 Flux Measurement: CModel

CModel fits a PSF-convolved galaxy model to measure **total flux**.

### Algorithm (from `lsst.meas.modelfit`)

**Three-stage fit:**

1. **Exponential fit** (Sérsic n=1, disk-like):
   Fit an elliptical exponential profile convolved with the PSF.
   Free parameters: centroid, ellipticity, half-light radius, flux.

2. **De Vaucouleurs fit** (Sérsic n=4, bulge-like):
   Same, but with a steeper profile.

3. **Linear combination:**
   Hold positions and shapes fixed from steps 1 & 2.
   Fit only the amplitude ratio: $f = f_{\text{fracDev}} \cdot f_{\text{deV}} + (1 - f_{\text{fracDev}}) \cdot f_{\text{exp}}$

This gives a bulge+disk decomposition. The combined flux captures
the total galaxy light better than any aperture.

**Output columns:**
```
modelfit_CModel_instFlux          # total CModel flux (instrumental)
modelfit_CModel_instFluxErr       # uncertainty
modelfit_CModel_fracDev           # fraction of flux in de Vaucouleurs component
modelfit_CModel_exp_instFlux      # exponential-only flux
modelfit_CModel_dev_instFlux      # de Vaucouleurs-only flux
```

### Why CModel for weak lensing?
- Galaxy **colours** from CModel fluxes across bands feed into photo-z
- CModel captures total flux, while apertures miss the wings
- `fracDev` gives a crude morphological classification (bulge vs. disk)

## 7.4 Forced Photometry

After measuring on the coadd, **forced photometry** goes back to the
individual visits and measures at the coadd-defined positions:

- Positions and shapes are **fixed** from the coadd catalog
- Only **fluxes** are fit on each single-visit image

This provides:
- **Consistent** multi-epoch photometry (same aperture in every visit)
- **Light curves** for variable objects
- Better photometry for faint objects (no detection noise)

```python
# Forced photometry via Butler:
forced_src = butler.get('forced_src', visit=903334, detector=16)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import galsim

rng_np = np.random.default_rng(42)

In [ ]:
# --- Demonstrate the full measurement chain on simulated galaxies ---

pixel_scale = 0.168  # arcsec/pixel (HSC)
psf_fwhm = 0.65      # arcsec (good seeing)
stamp_size = 64
noise_sigma = 25.0

psf = galsim.Moffat(beta=3.5, fwhm=psf_fwhm)  # realistic PSF
psf_image = psf.drawImage(nx=stamp_size, ny=stamp_size, scale=pixel_scale)

# Create a galaxy with known properties
true_hlr = 0.5       # arcsec
true_flux = 5e4
true_e1 = 0.15
true_e2 = -0.10
true_n = 1.5          # Sérsic index

gal = galsim.Sersic(n=true_n, half_light_radius=true_hlr, flux=true_flux)
gal = gal.shear(e1=true_e1, e2=true_e2)

# Apply a known shear
g1_applied = 0.02
g2_applied = -0.01
gal = gal.shear(g1=g1_applied, g2=g2_applied)

final = galsim.Convolve([gal, psf])
image = final.drawImage(nx=stamp_size, ny=stamp_size, scale=pixel_scale)
image.addNoise(galsim.GaussianNoise(galsim.BaseDeviate(42), sigma=noise_sigma))

In [ ]:
# Step 1: Measure adaptive moments (like base_SdssShape)
moments = galsim.hsm.FindAdaptiveMom(image)
psf_moments = galsim.hsm.FindAdaptiveMom(psf_image)

print("=== Adaptive Moments (SdssShape equivalent) ===")
print(f"Galaxy moments:  e1={moments.observed_shape.e1:.4f}, e2={moments.observed_shape.e2:.4f}")
print(f"Galaxy size:     σ={moments.moments_sigma:.3f} pixels")
print(f"PSF moments:     e1={psf_moments.observed_shape.e1:.4f}, e2={psf_moments.observed_shape.e2:.4f}")
print(f"PSF size:        σ={psf_moments.moments_sigma:.3f} pixels")
print()

# Resolution factor
T_gal = moments.moments_sigma**2
T_psf = psf_moments.moments_sigma**2
R = 1 - T_psf / T_gal
print(f"Resolution factor R = {R:.3f}  (>{0.3:.1f} is well-resolved)")
print()

# Step 2: PSF-corrected shapes (REGAUSS)
hsm_result = galsim.hsm.EstimateShear(image, psf_image, shear_est='REGAUSS')
print("=== REGAUSS (PSF-corrected) ===")
print(f"Corrected e1 = {hsm_result.corrected_e1:.4f}")
print(f"Corrected e2 = {hsm_result.corrected_e2:.4f}")
print(f"Corrected σ  = {hsm_result.corrected_shape_err:.4f}")
print()
print(f"True intrinsic e1={true_e1}, e2={true_e2}")
print(f"(Plus applied shear g1={g1_applied}, g2={g2_applied})")

In [ ]:
# Step 3: Compare all four HSM methods

methods = ['REGAUSS', 'KSB', 'BJ', 'LINEAR']
results = {}

print(f"{'Method':<10} {'e1':>8} {'e2':>8} {'σ_e':>8}")
print('=' * 38)

for method in methods:
    try:
        res = galsim.hsm.EstimateShear(image, psf_image, shear_est=method)
        results[method] = (res.corrected_e1, res.corrected_e2)
        print(f"{method:<10} {res.corrected_e1:>8.4f} {res.corrected_e2:>8.4f} {res.corrected_shape_err:>8.4f}")
    except Exception as e:
        print(f"{method:<10} FAILED: {e}")

print(f"\n{'True':>10} {true_e1:>8.4f} {true_e2:>8.4f}")
print(f"(intrinsic only — shear adds g1={g1_applied}, g2={g2_applied})")

In [ ]:
# Visualize: the galaxy, PSF, and shape measurement

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Galaxy image
im = image.array
axes[0].imshow(im, cmap='viridis', origin='lower')
axes[0].set_title('Galaxy (PSF-convolved + noise)')

# PSF
axes[1].imshow(psf_image.array, cmap='inferno', origin='lower')
axes[1].set_title(f'PSF (FWHM={psf_fwhm}")')

# Noiseless model (what CModel tries to fit)
noiseless = final.drawImage(nx=stamp_size, ny=stamp_size, scale=pixel_scale)
axes[2].imshow(noiseless.array, cmap='viridis', origin='lower')
axes[2].set_title('Noiseless model')

# Residual (data - model)
residual = im - noiseless.array
axes[3].imshow(residual, cmap='RdBu_r', origin='lower',
               vmin=-3*noise_sigma, vmax=3*noise_sigma)
axes[3].set_title('Residual (noise only)')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Shape Measurement: Galaxy → PSF correction → Ellipticity',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/measurement_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Demonstrate the effect of PSF on shape measurement ---

# Vary PSF FWHM and show how resolution degrades
fwhm_values = np.linspace(0.3, 1.5, 20)
e1_measured = []
R_values = []

for fwhm in fwhm_values:
    test_psf = galsim.Moffat(beta=3.5, fwhm=fwhm)
    test_final = galsim.Convolve([gal, test_psf])
    test_img = test_final.drawImage(nx=stamp_size, ny=stamp_size, scale=pixel_scale)
    test_img.addNoise(galsim.GaussianNoise(galsim.BaseDeviate(42), sigma=noise_sigma))
    test_psf_img = test_psf.drawImage(nx=stamp_size, ny=stamp_size, scale=pixel_scale)

    try:
        res = galsim.hsm.EstimateShear(test_img, test_psf_img, shear_est='REGAUSS')
        e1_measured.append(res.corrected_e1)

        gal_mom = galsim.hsm.FindAdaptiveMom(test_img)
        psf_mom = galsim.hsm.FindAdaptiveMom(test_psf_img)
        R_values.append(1 - psf_mom.moments_sigma**2 / gal_mom.moments_sigma**2)
    except:
        e1_measured.append(np.nan)
        R_values.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fwhm_values, R_values, 'o-', color='steelblue')
axes[0].axhline(0.3, color='red', ls='--', label='R=0.3 cut')
axes[0].set_xlabel('PSF FWHM (arcsec)', fontsize=12)
axes[0].set_ylabel('Resolution factor R', fontsize=12)
axes[0].set_title('Galaxy resolution vs. seeing')
axes[0].legend()

axes[1].plot(fwhm_values, e1_measured, 'o-', color='forestgreen')
axes[1].axhline(true_e1, color='red', ls='--', label=f'True e1={true_e1}')
axes[1].set_xlabel('PSF FWHM (arcsec)', fontsize=12)
axes[1].set_ylabel('Measured e1 (REGAUSS)', fontsize=12)
axes[1].set_title('REGAUSS recovery vs. seeing')
axes[1].legend()

plt.tight_layout()
plt.savefig('../../figures/resolution_vs_seeing.png', dpi=150, bbox_inches='tight')
plt.show()

print("As the PSF grows (worse seeing), resolution drops and")
print("shape measurement becomes noisier and more biased.")
print("This is why PSF modeling accuracy is paramount.")

## 7.5 The Output Catalog

The measurement step produces a catalog (`deepCoadd_meas`) with hundreds
of columns. The key ones for weak lensing:

### Shapes
```
ext_shapeHSM_HsmShapeRegauss_e1       # shear estimator e1
ext_shapeHSM_HsmShapeRegauss_e2       # shear estimator e2
ext_shapeHSM_HsmShapeRegauss_sigma    # corrected size
ext_shapeHSM_HsmShapeRegauss_flag     # quality flag
ext_shapeHSM_HsmSourceMoments_xx      # raw second moment Ixx
ext_shapeHSM_HsmSourceMoments_yy      # raw second moment Iyy
ext_shapeHSM_HsmSourceMoments_xy      # raw second moment Ixy
ext_shapeHSM_HsmPsfMoments_xx         # PSF model moment Ixx
ext_shapeHSM_HsmPsfMoments_yy         # PSF model moment Iyy
ext_shapeHSM_HsmPsfMoments_xy         # PSF model moment Ixy
```

### Fluxes
```
base_PsfFlux_instFlux                 # PSF flux (for stars)
modelfit_CModel_instFlux              # CModel total flux
modelfit_CModel_fracDev               # bulge fraction
base_CircularApertureFlux_*_instFlux  # aperture fluxes
```

### Quality / selection
```
detect_isPrimary                      # use this for clean catalog
base_ClassificationExtendedness_value # 0=star, 1=galaxy
deblend_skipped                       # deblending problems
base_PixelFlags_flag_*                # pixel quality flags
```

## Summary: From Pixels to Shapes

```
deepCoadd pixels
     │
     ▼
NoiseReplacer: isolate each deblended child
     │
     ├──► SdssCentroid: find center (x, y)
     │
     ├──► SdssShape: adaptive moments → raw (Ixx, Iyy, Ixy)
     │
     ├──► HsmShapeRegauss: PSF-corrected (e1, e2, σ) ← THIS IS THE SHEAR
     │
     ├──► PsfFlux: flux assuming point source
     │
     ├──► CModel: fit galaxy model → total flux
     │
     └──► Flags: quality indicators
           │
           ▼
     Source catalog (one row per source)
```

**Next:** [08_photoz.ipynb](08_photoz.ipynb) — Photometric redshifts